In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import glob
import pickle
import itertools
import numpy as np
from tqdm import tqdm

import evaluation

/gs/bs/tgh-25IAC/ud03523/TOOL/pydev/envs/jupyter/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the protocol
eval_data_path = Path('../data/protocols/eval.csv')
eval_data_list = pd.read_csv(eval_data_path, names=['data'])


# get the language label
eval_data_list['lang'] = eval_data_list['data'].apply(lambda x: x.split('_')[1])

# get the vocoder label
eval_data_list['attack'] = eval_data_list['data'].apply(lambda x: '_'.join(x.split('/')[0].split('_')[2:]))

eval_data_list.head()

,data,lang,attack
0,replacement_de_hifigan/vtt_modified/IxD9PhJ9aD...,de,hifigan
1,replacement_de_hifigan/vtt_modified/IxD9PhJ9aD...,de,hifigan
2,replacement_de_hifigan/vtt_modified/FkIcyFEbM_...,de,hifigan
3,replacement_de_hifigan/vtt_modified/6eNxVeS6Cv...,de,hifigan
4,replacement_de_hifigan/vtt_modified/hYq6KJnwPI...,de,hifigan


In [3]:
csv_save_dir = Path('project/results_csv')
csv_save_dir.mkdir(exist_ok=True)

In [ ]:
hparams = ['full_x01','full_x02','full_x03','full_x04','full_x05']

for hparam in hparams:
    # load results
    sys_path = Path('project/cp_whisper_hparams_{:s}.yaml/artifacts/outputs/'.format(hparam))

    output_files = glob.glob('*.pkl', root_dir=sys_path)
    assert len(output_files)==1, "Found multiple files, choose one to analysis: {:s}".format('\n'.join(output_files))

    with open(sys_path / output_files[0], 'rb') as file_ptr:
        output_data = pickle.load(file_ptr)
    assert len(output_data) == 2, "The input is supposed to be [res, refs]"
    output_data = [np.array(output_data[0]), np.array(output_data[1])]
    
    # comptue results
    langs = sorted(eval_data_list['lang'].unique())
    attacks = sorted(eval_data_list['attack'].unique())

    metrics_all = []
    for lang, attack in itertools.product(langs, attacks):
        data_idx = eval_data_list.query('lang == "{:s}" and attack == "{:s}"'.format(lang, attack)).index
        res, refs = output_data[0][data_idx], output_data[1][data_idx]
        cer, wer, metrics = evaluation.comptue_metrics(res, refs)
        metrics['cer'] = cer * 100
        metrics['wer'] = wer * 100
        metrics['lang'] = lang
        metrics['attack'] = attack
        metrics_all.append(metrics)
        
    # save
    results = pd.DataFrame(metrics_all)
    results.to_csv(csv_save_dir / 'cp_whisper_hparams_{:s}.csv'.format(hparam))
    print(csv_save_dir / 'cp_whisper_hparams_{:s}.csv'.format(hparam))

project/results_csv/cp_whisper_hparams_full_x01.csv
project/results_csv/cp_whisper_hparams_full_x02.csv
project/results_csv/cp_whisper_hparams_full_x03.csv
